# W\&B Confusion Matrix Provenance Notebook

- Original source: Derrick Chan-Sew
- Purpose: reconstruct the supplemental ViT confusion matrix from a Weights & Biases confusion-matrix table artifact
- Expected input artifact filename: `media_table_conf_mat_top1_inference_xrd_model_duninwhx_table_4_89d4c914203af59fa295.table.json`
- Note: exact rerun requires the correct model/run/artifact mapping used for the manuscript figure

This notebook is retained as a provenance artifact for convenience. It is not the primary paper-facing scripted workflow.


In [ ]:
from pathlib import Path

# Set this to the exported W&B table JSON for the confusion matrix you want to reproduce.
ARTIFACT_JSON = Path("media_table_conf_mat_top1_inference_xrd_model_duninwhx_table_4_89d4c914203af59fa295.table.json")
OUTPUT_PNG = Path("confusion_matrix.png")
OUTPUT_PNG_NORMALIZED = Path("confusion_matrix_normalized.png")
NUM_CLASSES = 99

if not ARTIFACT_JSON.exists():
    print(f"Set ARTIFACT_JSON to a local W&B table JSON before running. Current value: {ARTIFACT_JSON}")


In [ ]:
import json
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

with ARTIFACT_JSON.open("r") as f:
    data = json.load(f)

df = pd.DataFrame(data["data"], columns=data["columns"])
conf_matrix = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=float)
for _, row in df.iterrows():
    actual = int(row["Actual"])
    predicted = int(row["Predicted"])
    weight = float(row["nPredictions"])
    conf_matrix[actual, predicted] += weight

plt.figure(figsize=(16, 12))
sns.heatmap(conf_matrix, cmap="Blues", annot=False)
plt.title("ViT Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.savefig(OUTPUT_PNG, bbox_inches="tight")
plt.show()


In [ ]:
row_sums = conf_matrix.sum(axis=1, keepdims=True)
conf_matrix_norm = np.divide(
    conf_matrix,
    row_sums,
    out=np.zeros_like(conf_matrix),
    where=row_sums != 0,
)

plt.figure(figsize=(20, 16))
sns.heatmap(conf_matrix_norm, cmap="viridis", annot=False, cbar=True)
plt.title("ViT Confusion Matrix (Row-normalized)")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.savefig(OUTPUT_PNG_NORMALIZED, bbox_inches="tight")
plt.show()
